In [ ]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv


ModuleNotFoundError: No module named 'google.generativeai'

In [2]:
from google import genai
from google.genai import types

In [3]:
# 1. Load API Key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# 2. Khởi tạo Client
client = genai.Client(api_key=GOOGLE_API_KEY)

In [11]:
# 3. Định nghĩa đường dẫn
# Input file
input_file_path = Path(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf")

# Lấy tên thư mục cha của file (ví dụ: "Ngoai_ngu")
subfolder_name = input_file_path.parent.name

# Thư mục gốc cho dữ liệu đã tiền xử lý
base_output_dir = Path(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Preprocessed_Data")

# Tạo đường dẫn đầy đủ: Preprocessed_Data/Ngoai_ngu
final_output_folder = base_output_dir / subfolder_name

# Đảm bảo thư mục đích tồn tại (tạo mới nếu chưa có)
final_output_folder.mkdir(parents=True, exist_ok=True)

# Tên file đầu ra
output_file_path = final_output_folder / f"{input_file_path.stem}_preprocessed.txt"

In [16]:
# 4. Thiết lập System Prompt
system_instruction = """
Bạn là chuyên gia tiền xử lý dữ liệu RAG cho Quy chế đào tạo.
Tôi gửi bạn một file pdf hoặc word có chứa nội dung liên quan đến Quy chế đào tạo.
Nhiệm vụ: Tiền xử lý (loại bỏ lỗi đánh máy, lỗi trình bày...) và chia tài liệu thành các chunk logic. 
LƯU Ý: Chỉ trả về dữ liệu nằm trong các thẻ <<<CHUNK>>>. 
KHÔNG giải thích, KHÔNG chào hỏi, KHÔNG thêm text thừa ở đầu/cuối response.

Định dạng mỗi chunk:

<<<CHUNK>>>
source: Ngoai_ngu_2022_Quy_doi_CCTA.pdf
parent_doc_id: Ngoai_ngu_2022_Quy_doi_CCTA
chunk_id: [parent_doc_id]_[số thứ tự 001, 002...]
chunk_index: [số thứ tự bắt đầu từ 1]
language: vi
category: Ngoai_ngu
year: 2022
title: [Tiêu đề chunk]
summary: [Tóm tắt 2-3 câu ngắn gọn, đảm bảo đầy đủ từ khóa phù hợp cho retrieval]

[CONTENT]
(Nội dung chi tiết, nếu là text thì trả về text, nếu là bảng biểu thì trả về dạng markdown của nó, nếu là ảnh thì trả về mô tả chi tiết)
<<<CHUNK>>>
"""

In [23]:
def process_document():
    print(f"--- Đang upload file PDF: {input_file_path.name} ---")
    
    # Upload file lên Gemini File API
    file_upload = client.files.upload(file=input_file_path)
    
    # Chờ file được xử lý xong trên server (PDF cần thời gian OCR/Parsing)
    while file_upload.state == "PROCESSING":
        print(".", end="", flush=True)
        time.sleep(2)
        file_upload = client.files.get(name=file_upload.name)
    
    print(f"\n--- File ready! Đang gửi yêu cầu cho Gemini 3.1 Flash Lite ---")

    # Gửi yêu cầu kèm file đã upload
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite", 
        contents=[file_upload],
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.1
        )
    )
    
    # Logic loại bỏ text thừa (nếu có)
    raw_text = response.text
    if "<<<CHUNK>>>" in raw_text:
        clean_content = "<<<CHUNK>>>" + raw_text.split("<<<CHUNK>>>", 1)[-1]
    else:
        clean_content = raw_text

    # 4. Lưu kết quả
    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write(clean_content.strip())
    
    # Xóa file trên server sau khi xử lý xong (optional)
    client.files.delete(name=file_upload.name)
    
    print(f"--- Hoàn thành! File lưu tại: {output_file_path} ---")

In [24]:
process_document()

--- Đang upload file PDF: Ngoai_ngu_2022_Quy_doi_CCTA.pdf ---

--- File ready! Đang gửi yêu cầu cho Gemini 3.1 Flash Lite ---
--- Hoàn thành! File lưu tại: C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Preprocessed_Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA_preprocessed.txt ---
